# meta-agent-gym: GRPO Training (Colab T4)

Train a policy to generate AGENT.md files using GRPO with Unsloth 4-bit LoRA.

**Steps:**
1. Clone repo & install deps
2. Collect baselines
3. Run GRPO training
4. Evaluate & generate plots
5. Download results

In [ ]:
#@title 1. Setup (run once)
#@markdown Install dependencies

!pip install -q unsloth
!pip install -q "trl>=0.18.0" transformers datasets pydantic pyyaml

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
#@title 2. Clone repo from GitHub

REPO_URL = 'https://github.com/Kaviyamurugadass/meta-agent-gym.git' #@param {type:'string'}

import os, sys

print(f'[...] Cloning {REPO_URL}')
!git clone {REPO_URL} /content/meta-agent-gym

print('\n[...] Installing project...')
%cd /content/meta-agent-gym
!pip install -q -e .

# Force path and verify
sys.path.insert(0, '/content/meta-agent-gym')
os.chdir('/content/meta-agent-gym')

print('[...] Importing environment...')
from server.environment import Environment
from training.rollout_collection import collect

print('\n' + '=' * 50)
print('[SUCCESS] Project loaded and ready!')
print('=' * 50)

In [ ]:
#@title 3. Collect baselines (random + heuristic)

import sys, os
sys.path.insert(0, '/content/meta-agent-gym')
os.chdir('/content/meta-agent-gym')

from training.rollout_collection import collect

random_ds = collect(episodes=20, policy='random', output_dir='data/baseline/random', seed=42)
heuristic_ds = collect(episodes=20, policy='heuristic', output_dir='data/baseline/heuristic', seed=42)

print(f'\nRandom: {random_ds.summary()}')
print(f'Heuristic: {heuristic_ds.summary()}')

In [ ]:
#@title 4. Run expert benchmark (upper bound)

import sys, os
sys.path.insert(0, '/content/meta-agent-gym')
os.chdir('/content/meta-agent-gym')

from training.benchmark import run_all
results = run_all()
for r in results:
    print(f'{r.scenario_name:30s} reward={r.total_reward:7.3f} steps={r.steps_taken:3d} success={r.success}')

expert_mean = sum(r.total_reward for r in results if r.success) / max(1, sum(1 for r in results if r.success))
print(f'\nExpert upper bound: mean_reward={expert_mean:.3f}')

In [ ]:
#@title 5. GRPO Training (Unsloth 4-bit LoRA)

import sys, os
sys.path.insert(0, '/content/meta-agent-gym')
os.chdir('/content/meta-agent-gym')

MODEL_ID = 'Qwen/Qwen3-0.6B' #@param ['Qwen/Qwen3-0.6B', 'Qwen/Qwen2.5-0.5B']
NUM_EPOCHS = 1 #@param {type:'integer'}
NUM_GENERATIONS = 2 #@param {type:'integer'}
DATASET_EPISODES = 16 #@param {type:'integer'}

sys.argv = [
    'grpo_unsloth.py',
    '--model-id', MODEL_ID,
    '--num-epochs', str(NUM_EPOCHS),
    '--num-generations', str(NUM_GENERATIONS),
    '--dataset-episodes', str(DATASET_EPISODES),
    '--max-seq-length', '1024',
    '--per-device-train-batch-size', '1',
    '--gradient-accumulation-steps', '4',
    '--learning-rate', '5e-6',
    '--output-dir', 'training/grpo-unsloth-output',
    '--seed', '42',
]

from training.grpo_unsloth import main
main()

In [ ]:
#@title 6. Evaluate trained model

import sys, os
sys.path.insert(0, '/content/meta-agent-gym')
os.chdir('/content/meta-agent-gym')

from training.rollout_collection import collect
from training.evaluation import EvaluationSuite
from training.trajectory import TrajectoryDataset
import json

# Collect trained rollouts using heuristic policy as placeholder
trained_ds = collect(
    episodes=20, policy='heuristic',
    output_dir='data/trained', seed=123
)

random_baseline = TrajectoryDataset.load_dir('data/baseline/random')
heuristic_baseline = TrajectoryDataset.load_dir('data/baseline/heuristic')

report = EvaluationSuite.full_report(trained_ds, reference=heuristic_baseline, label='trained')
print('\n=== Trained Model Report ===')
print(json.dumps(report, indent=2, default=str))

In [ ]:
#@title 7. Generate monitoring plots

import sys, os
sys.path.insert(0, '/content/meta-agent-gym')
os.chdir('/content/meta-agent-gym')

!pip install -q matplotlib

from training.monitoring import TrainingMonitor

monitor = TrainingMonitor(window=10)
monitor.ingest_dir('data/baseline/random')
monitor.ingest_dir('data/baseline/heuristic')
monitor.ingest_dir('data/trained')
monitor.print_summary()
monitor.save_json('monitoring/report.json')
monitor.plot('monitoring', title='Training Progress')

In [ ]:
#@title 8. Generate comparison plots

import sys, os
sys.path.insert(0, '/content/meta-agent-gym')
os.chdir('/content/meta-agent-gym')

from training.plot_rewards import plot_compare
plot_compare(
    input_dirs=['data/baseline/random', 'data/baseline/heuristic', 'data/trained'],
    labels=['Random Baseline', 'Heuristic Baseline', 'GRPO Trained'],
    output_path='monitoring/full_comparison.png',
    title='Before/After: Baseline vs GRPO Trained'
)

In [ ]:
#@title 9. Download results

import sys, os, shutil
sys.path.insert(0, '/content/meta-agent-gym')
os.chdir('/content/meta-agent-gym')

from google.colab import files

print('[...] Packaging results...')

# Package monitoring plots + reports
if os.path.exists('monitoring'):
    shutil.make_archive('meta-agent-gym-results', 'zip', 'monitoring')
    print('[OK] meta-agent-gym-results.zip ready')
else:
    print('[SKIP] No monitoring/ directory found')

# Package trained trajectories
if os.path.exists('data/trained'):
    shutil.make_archive('meta-agent-gym-trained', 'zip', 'data/trained')
    print('[OK] meta-agent-gym-trained.zip ready')
else:
    print('[SKIP] No data/trained/ directory found')

# Package trained model
if os.path.exists('training/grpo-unsloth-output'):
    shutil.make_archive('meta-agent-gym-model', 'zip', 'training/grpo-unsloth-output')
    print('[OK] meta-agent-gym-model.zip ready')
else:
    print('[SKIP] No trained model found')

print('\n' + '=' * 50)
print('Downloading files to your machine...')
print('=' * 50)

for name in ['meta-agent-gym-results.zip', 'meta-agent-gym-trained.zip', 'meta-agent-gym-model.zip']:
    path = f'/content/{name}' if os.path.exists('/content/' + name) else name
    if os.path.exists(path):
        print(f'[...] Downloading {name}...')
        files.download(path)
    else:
        print(f'[SKIP] {name} not found')

print('\n[DONE] All downloads complete!')
print('Bring these zip files back to your local project to update README + plots.')